# Importações e leitura da base HubSpot
Este bloco importa bibliotecas essenciais, define o caminho do arquivo CSV do HubSpot e tenta carregar os dados usando `pd.read_csv` com `encoding='utf-8'` e `low_memory=False`. Mostra as 5 primeiras linhas e a lista de colunas; trata `FileNotFoundError` e exceções gerais.

In [ ]:
import pandas as pd
import os

# --- Caminhos ---
caminho_hubspot = r"C:\Users\a483650\Projetos\_DADOS_CENTRALIZADOS\hubspot\hubspot_leads_ATUAL.csv"

# --- Execução ---
try:
    print(f"Lendo a base de dados do HubSpot com encoding 'utf-8'...")
    # Usando o parâmetro low_memory=False para evitar alertas de tipos mistos em colunas grandes
    df = pd.read_csv(caminho_hubspot, encoding='utf-8', low_memory=False)
    print("Leitura concluída com sucesso!")
    
    print("\nAs 5 primeiras linhas da base são:")
    # Usa safe_display para compatibilidade com ambientes sem nbformat recente
    # safe_display será definido na próxima célula
    safe_display(df.head())
    
    print("\nColunas disponíveis:")
    print(df.columns.tolist())

except FileNotFoundError:
    print(f"ERRO: O arquivo de origem não foi encontrado em '{caminho_hubspot}'")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

# Seleção, renomeação e conversão de colunas
Este bloco cria um dicionário de mapeamento de nomes de colunas, verifica colunas faltantes, filtra apenas as colunas existentes, renomeia-as para nomes padronizados e converte campos de data (`Data_Criacao`, `Data_Fechamento`) para `datetime`. Remove linhas sem `Data_Criacao` e exibe uma amostra e os tipos de dados.
# Amostragem e preview da base final
Abaixo geramos uma amostra rápida para inspeção (até 500 linhas) e verificamos o formato final que será exportado para o Looker.

In [ ]:
# Utilitários: safe_display para fallback quando nbformat/renderer não estiver disponível
from IPython.display import display as _display
def safe_display(obj, max_rows=20):
    try:
        _display(obj)
    except Exception as e:
        # fallback para pandas DataFrame/Series
        try:
            import pandas as _pd
            if isinstance(obj, _pd.DataFrame) or isinstance(obj, _pd.Series):
                print(obj.head(max_rows).to_string())
            else:
                print(obj)
        except Exception:
            try:
                print(obj)
            except Exception:
                print(repr(obj))
        print(f'NOTE: display failed with: {e}')

# Dicionário de mapeamento com os nomes corretos das colunas
colunas_interesse = {
    'Record ID': 'ID_Negocio',
    'Nome do negócio': 'Nome_Negocio',
    'Data de criação': 'Data_Criacao',
    'Etapa do negócio': 'Etapa_Negocio',
    'Fonte original do tráfego': 'Fonte_Trafego',
    'Detalhamento da fonte original do tráfego 1': 'Fonte_Detalhe_1',
    'Detalhamento da fonte original do tráfego 2': 'Fonte_Detalhe_2',
    'Data de fechamento': 'Data_Fechamento',
    'Unidade Desejada': 'Unidade'
}

# Verifica se todas as colunas de interesse existem no DataFrame
colunas_existentes = [col for col in colunas_interesse.keys() if col in df.columns]
colunas_faltantes = [col for col in colunas_interesse.keys() if col not in df.columns]

if colunas_faltantes:
    print(f"Atenção: As seguintes colunas não foram encontradas na base: {colunas_faltantes}")

print(f"Colunas que serão utilizadas: {colunas_existentes}")

# Filtra apenas as colunas de interesse que existem
df_filtrado = df[colunas_existentes].copy() # Usar .copy() para evitar SettingWithCopyWarning

# Renomeia as colunas
df_renomeado = df_filtrado.rename(columns=colunas_interesse)

# Converte colunas de data para o formato datetime
# O formato da data no CSV é 'YYYY-MM-DD HH:MM:SS' ou similar, o to_datetime padrão deve funcionar
df_renomeado['Data_Criacao'] = pd.to_datetime(df_renomeado['Data_Criacao'], errors='coerce')
if 'Data_Fechamento' in df_renomeado.columns:
    df_renomeado['Data_Fechamento'] = pd.to_datetime(df_renomeado['Data_Fechamento'], errors='coerce')

# Remove linhas onde a Data_Criacao é nula após a conversão
df_renomeado.dropna(subset=['Data_Criacao'], inplace=True)

# Exibe as primeiras linhas e os tipos de dados para verificação
print("\nDataFrame após limpeza e renomeação:")
safe_display(df_renomeado.head())

print("\nTipos de dados das colunas:")
df_renomeado.info()

# Enriquecimento e features temporais
Normaliza valores de `Unidade`, `Fonte_Trafego` e `Etapa_Negocio` preenchendo nulos e removendo espaços. Cria colunas temporais (`Ano`, `Mes`, `Semana`) e uma coluna booleana `_is_converted` baseada em uma heurística de palavras-chave para identificar negócios convertidos/ganhos. Exibe amostra final e contagens.

In [ ]:
print("Enriquecendo o DataFrame com novas colunas para análise...\n")

# --- Normalização de Colunas ---
# Preenche valores nulos e remove espaços em branco para consistência nos agrupamentos
if 'Unidade' in df_renomeado.columns:
    df_renomeado['Unidade'] = df_renomeado['Unidade'].fillna('Sem Unidade').str.strip()
else:
    df_renomeado['Unidade'] = 'Sem Unidade'

if 'Fonte_Trafego' in df_renomeado.columns:
    df_renomeado['Fonte_Trafego'] = df_renomeado['Fonte_Trafego'].fillna('Sem Fonte').str.strip()
else:
    df_renomeado['Fonte_Trafego'] = 'Sem Fonte'
    
if 'Etapa_Negocio' in df_renomeado.columns:
    df_renomeado['Etapa_Negocio'] = df_renomeado['Etapa_Negocio'].fillna('Sem Etapa').str.strip()
else:
    df_renomeado['Etapa_Negocio'] = 'Sem Etapa'


# --- Features de Data ---
df_renomeado['Ano'] = df_renomeado['Data_Criacao'].dt.year
# Usando to_period('M') para garantir que o mês seja o primeiro dia do mês, facilitando o group by
df_renomeado['Mes'] = df_renomeado['Data_Criacao'].dt.to_period('M').apply(lambda r: r.start_time)
# Calcula o início da semana (considerando domingo como início)
df_renomeado['Semana'] = (df_renomeado['Data_Criacao'] - pd.to_timedelta((df_renomeado['Data_Criacao'].dt.weekday + 1) % 7, unit='D')).dt.normalize()


# --- Definição de Conversão ---
# Heurística para identificar negócios ganhos/convertidos
final_kw = ['won', 'fechado', 'convertido', 'cliente', 'closed won', 'closedwon', 'negócio ganho']
df_renomeado['_is_converted'] = df_renomeado['Etapa_Negocio'].astype(str).str.lower().apply(lambda s: any(k in s for k in final_kw))


print("DataFrame enriquecido com sucesso. Amostra dos dados:")
safe_display(df_renomeado.head())

print("\nVerificando a nova coluna de conversão:")
print(df_renomeado[['Etapa_Negocio', '_is_converted']].value_counts().head(10))

In [ ]:
# Filtrar para Unidades Próprias (lista fornecida)
unidades_proprias = ['VILA LEOPOLDINA','MORUMBI','ITAIM BIBI','PACAEMBU','PINHEIROS','JARDINS','PERDIZES','SANTANA']
# Normaliza nomes para garantir correspondência
if 'Unidade' not in df_renomeado.columns:
    print("A coluna 'Unidade' não existe na base. Verifique mapeamentos.")
else:
    df_renomeado['Unidade_normalizada'] = df_renomeado['Unidade'].astype(str).str.upper().str.strip()
    unidades_norm = [u.upper().strip() for u in unidades_proprias]
    before_n = len(df_renomeado)
    df_renomeado = df_renomeado[df_renomeado['Unidade_normalizada'].isin(unidades_norm)].copy()
    after_n = len(df_renomeado)
    print(f'Filtradas Unidades Próprias: {len(unidades_proprias)} unidades. Linhas antes: {before_n}, depois: {after_n}')
    # Exibe as unidades que ficaram na base
    safe_display(df_renomeado['Unidade'].value_counts())
    # Remove coluna auxiliar usada para normalização
    df_renomeado.drop(columns=['Unidade_normalizada'], inplace=True)


In [ ]:
# Importa plotly quando disponível, senão prossegue sem plotly
try:
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

# Amostra para inspeção (até 500 linhas)
n = min(500, len(df_renomeado))
df_sample = df_renomeado.sample(n=n, random_state=1) if n>0 else df_renomeado.head(0)
print('Tamanho da base final:', df_renomeado.shape)
print('Tamanho da amostra:', df_sample.shape)
safe_display(df_sample.head())

## Diagnóstico: garantir colunas temporais
Esta célula verifica/normaliza nomes de colunas, converte `Data_Criacao` para datetime e (re)cria `Ano`, `Mes` e `Semana` caso estejam ausentes ou com valores inválidos. Rode antes das agregações.

In [ ]:
# Diagnóstico e criação de colunas temporais necessárias
import pandas as pd
# Normaliza nomes de colunas (remove espaços acidentais)
if 'df_renomeado' not in globals():
    if 'df' in globals():
        df_renomeado = df.copy()
        print("Aviso: usando 'df' como fallback para df_renomeado.")
    else:
        raise NameError("Nem 'df_renomeado' nem 'df' estão definidos. Execute a leitura/renomeação primeiro.")
df_renomeado.columns = df_renomeado.columns.astype(str).str.strip()
# Verifica existência de Data_Criacao
if 'Data_Criacao' not in df_renomeado.columns:
    raise KeyError("Coluna 'Data_Criacao' não encontrada em df_renomeado. Verifique nomes e mapeamentos.")
# Converte para datetime de forma robusta
df_renomeado['Data_Criacao'] = pd.to_datetime(df_renomeado['Data_Criacao'], errors='coerce')
n_null_dates = df_renomeado['Data_Criacao'].isna().sum()
print(f'Conversão Data_Criacao completa — nulos: {n_null_dates} de {len(df_renomeado)}')
# Cria/atualiza colunas temporais
df_renomeado['Ano'] = df_renomeado['Data_Criacao'].dt.year
df_renomeado['Mes'] = df_renomeado['Data_Criacao'].dt.to_period('M').apply(lambda r: r.start_time)
df_renomeado['Semana'] = (df_renomeado['Data_Criacao'] - pd.to_timedelta((df_renomeado['Data_Criacao'].dt.weekday + 1) % 7, unit='D')).dt.normalize()
print('Colunas temporais garantidas: ', [c for c in ['Ano','Mes','Semana'] if c in df_renomeado.columns])
safe_display(df_renomeado[['Data_Criacao','Mes','Semana']].head())

In [ ]:
# Leads por semana (geral)
leads_week = df_renomeado.groupby('Semana').size().reset_index(name='Leads').sort_values('Semana')
safe_display(leads_week.head())
fig = px.line(leads_week, x='Semana', y='Leads', title='Leads por Semana (Geral)')
fig.show()

# Leads por mês (geral)
leads_month = df_renomeado.groupby('Mes').size().reset_index(name='Leads').sort_values('Mes')
safe_display(leads_month.head())
fig2 = px.bar(leads_month, x='Mes', y='Leads', title='Leads por Mês (Geral)')
fig2.update_xaxes(type='category')
fig2.show()

# Leads por mês por unidade (exemplo de pivot para validação)
leads_month_unit = df_renomeado.groupby(['Mes','Unidade']).size().reset_index(name='Leads')
safe_display(leads_month_unit.head())
# Mostra gráfico para as top 6 unidades por volume no mês mais recente
if not leads_month_unit.empty:
    latest_month = leads_month['Mes'].max()
    top_units = (leads_month_unit[leads_month_unit['Mes']==latest_month].groupby('Unidade')['Leads'].sum().nlargest(6).index.tolist())
    df_top = leads_month_unit[(leads_month_unit['Mes']==latest_month) & (leads_month_unit['Unidade'].isin(top_units))]
    fig3 = px.bar(df_top, x='Unidade', y='Leads', color='Unidade', title=f'Leads por Unidade no mês {latest_month}')
    fig3.show()

# Fonte de origem e Etapa do negócio
Contagem por canal (geral e por unidade) e resumo das etapas do negócio.

In [ ]:
# Contagem por Fonte (geral)
fonte_counts = df_renomeado.groupby('Fonte_Trafego').size().reset_index(name='Leads').sort_values('Leads', ascending=False)
safe_display(fonte_counts.head(20))
fig4 = px.bar(fonte_counts.head(20), x='Fonte_Trafego', y='Leads', title='Top 20 Fontes de Tráfego (Geral)')
fig4.update_xaxes(tickangle=45)
fig4.show()

# Contagem por Fonte e Unidade (exemplo)
fonte_unit = df_renomeado.groupby(['Fonte_Trafego','Unidade']).size().reset_index(name='Leads')
safe_display(fonte_unit.head())

# Resumo das etapas do negócio
etapa_counts = df_renomeado.groupby('Etapa_Negocio').size().reset_index(name='Leads').sort_values('Leads', ascending=False)
safe_display(etapa_counts.head(20))

# Taxa de conversão por canal (geral e por unidade)
Usamos a coluna `_is_converted` para calcular taxa = convertidos / total.

In [ ]:
# Conversão por Fonte (geral)
conv = df_renomeado.groupby('Fonte_Trafego')['_is_converted'].agg(['sum','count']).reset_index()
conv.rename(columns={'sum':'Converted','count':'Total'}, inplace=True)
conv['Conversion_Rate'] = conv['Converted'] / conv['Total']
conv = conv.sort_values('Conversion_Rate', ascending=False)
safe_display(conv.head(30))
fig5 = px.bar(conv.sort_values('Conversion_Rate', ascending=False).head(20), x='Fonte_Trafego', y='Conversion_Rate', title='Taxa de Conversão por Fonte (Top 20)')
fig5.update_yaxes(tickformat='.0%')
fig5.update_xaxes(tickangle=45)
fig5.show()

# Conversão por Fonte e Unidade (exemplo de tabela)
conv_unit = df_renomeado.groupby(['Unidade','Fonte_Trafego'])['_is_converted'].agg(['sum','count']).reset_index()
conv_unit.rename(columns={'sum':'Converted','count':'Total'}, inplace=True)
conv_unit['Conversion_Rate'] = conv_unit['Converted'] / conv_unit['Total']
safe_display(conv_unit.head(30))

# Preparar e exportar a base final para o Looker
Geramos uma tabela 'plana' com as colunas que normalmente alimentam o Looker e exportamos para Excel no diretório de `geracao_dashboard/outputs`.

In [ ]:
# Colunas finais esperadas - ajuste conforme necessidade do Looker
cols_final = [
    'ID_Negocio','Nome_Negocio','Data_Criacao','Data_Fechamento',
    'Unidade','Fonte_Trafego','Fonte_Detalhe_1','Fonte_Detalhe_2',
    'Etapa_Negocio','Ano','Mes','Semana','_is_converted'
]
# Garante que apenas colunas existentes sejam selecionadas
cols_final_existing = [c for c in cols_final if c in df_renomeado.columns]
df_export = df_renomeado[cols_final_existing].copy()
# Caminho de saída absoluto conforme solicitado
output_path = r'C:\Users\a483650\Projetos\03_Analises_Operacionais\Funil completo das unidades próprias\geracao_dashboard\outputs\hubspot_dashboard_base.xlsx'
import os
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# --- Preparar tabelas consolidadas e visão geral ---
# Resumo por unidade (consolidado = soma/contagem por unidade)
if 'Unidade' in df_export.columns:
    df_unit_summary = df_export.groupby('Unidade').agg(
        Leads=('ID_Negocio', 'count') if 'ID_Negocio' in df_export.columns else ('Data_Criacao','count'),
        Converted=('_is_converted', 'sum') if '_is_converted' in df_export.columns else ('Data_Criacao','count')
    ).reset_index()
    df_unit_summary['Conversion_Rate'] = df_unit_summary['Converted'] / df_unit_summary['Leads']
    # Totais gerais
    totals = pd.DataFrame({
        'Unidade': ['TOTAL'],
        'Leads': [df_unit_summary['Leads'].sum()],
        'Converted': [df_unit_summary['Converted'].sum()]
    })
    totals['Conversion_Rate'] = totals['Converted'] / totals['Leads']
    df_consolidado = pd.concat([df_unit_summary, totals], ignore_index=True)
else:
    df_consolidado = pd.DataFrame()

# Agregações temporais e por fonte/etapa para possíveis análises rápidas
agg_semanal = df_export.groupby('Semana').size().reset_index(name='Leads').sort_values('Semana') if 'Semana' in df_export.columns else pd.DataFrame()
agg_mensal = df_export.groupby('Mes').size().reset_index(name='Leads').sort_values('Mes') if 'Mes' in df_export.columns else pd.DataFrame()
agg_fonte = df_export.groupby('Fonte_Trafego').size().reset_index(name='Leads').sort_values('Leads', ascending=False) if 'Fonte_Trafego' in df_export.columns else pd.DataFrame()
agg_etapa = df_export.groupby('Etapa_Negocio').size().reset_index(name='Leads').sort_values('Leads', ascending=False) if 'Etapa_Negocio' in df_export.columns else pd.DataFrame()

# Visão geral / KPIs
total_leads = len(df_export)
total_converted = int(df_export['_is_converted'].sum()) if '_is_converted' in df_export.columns else 0
overall_conversion = total_converted / total_leads if total_leads>0 else 0
visao_metrics = pd.DataFrame({
    'Metric': ['Total Leads', 'Total Converted', 'Overall Conversion Rate'],
    'Value': [total_leads, total_converted, overall_conversion]
})
# Top fontes para visão
top_fontes = agg_fonte.head(10) if not agg_fonte.empty else pd.DataFrame()

# Exporta múltiplas abas: base_all, consolidado, visao_geral, por unidade e tabelas auxiliares
try:
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        # Aba base com todos os registros
        df_export.to_excel(writer, sheet_name='base_all', index=False)
        # Aba consolidado (soma/agregação por unidade)
        if not df_consolidado.empty:
            df_consolidado.to_excel(writer, sheet_name='consolidado', index=False)
        # Aba visao_geral com KPIs e top fontes abaixo
        visao_metrics.to_excel(writer, sheet_name='visao_geral', index=False, startrow=0)
        if not top_fontes.empty:
            top_fontes.to_excel(writer, sheet_name='visao_geral', index=False, startrow=len(visao_metrics)+3)
        # Abas por unidade
        if 'Unidade' in df_export.columns:
            for unit in sorted(df_export['Unidade'].dropna().unique()):
                sheet_name = str(unit)[:31]
                sheet_name = sheet_name.replace('/', '_').replace('\\', '_')
                df_unit = df_export[df_export['Unidade'] == unit].copy()
                if df_unit.empty:
                    continue
                df_unit.to_excel(writer, sheet_name=sheet_name, index=False)
        # Abas auxiliares de agregação temporal e por fonte/etapa
        if not agg_semanal.empty:
            agg_semanal.to_excel(writer, sheet_name='consolidado_semanal', index=False)
        if not agg_mensal.empty:
            agg_mensal.to_excel(writer, sheet_name='consolidado_mensal', index=False)
        if not agg_fonte.empty:
            agg_fonte.to_excel(writer, sheet_name='consolidado_fonte', index=False)
        if not agg_etapa.empty:
            agg_etapa.to_excel(writer, sheet_name='consolidado_etapa', index=False)
    print(f'Export concluído (Excel com abas por unidade + consolidado + visao_geral): {output_path}')
except Exception as e:
    print(f'Erro ao exportar para Excel: {e}. Tentando export CSV de fallback...')
    try:
        df_export.to_csv(output_path.replace('.xlsx', '.csv'), index=False)
        if not df_consolidado.empty:
            df_consolidado.to_csv(output_path.replace('.xlsx', '_consolidado.csv'), index=False)
        print('Export em CSV concluído como fallback.')
    except Exception as e2:
        print(f'Falha no fallback CSV: {e2}')

safe_display(df_export.head())